In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_set = pd.read_csv("../data/en_smiler_train.tsv", sep="\t")
test_set = pd.read_csv("../data/en_smiler_test.tsv", sep="\t")

In [ ]:
relations, counts = np.unique(train_set["label"], return_counts = True)
indexes = np.argsort(counts)[::-1]
print(f"{relations.shape[0]} relations :")
for i in indexes:
    r, c = relations[i], counts[i]
    print(f"{r} : {c}")

36 relations :
birth-place : 21560
has-type : 21560
is-where : 21560
movie-has-director : 21560
from-country : 21560
has-author : 21560
has-genre : 21560
has-occupation : 21560
headquarters : 14225
org-has-member : 13014
has-population : 12985
is-member-of : 12967
org-has-founder : 6114
has-parent : 5502
has-spouse : 3802
won-award : 3622
org-leader : 3462
has-edu : 3269
has-nationality : 2770
starring : 2656
has-child : 1939
event-year : 1896
no_relation : 1315
has-sibling : 715
has-length : 633
has-lifespan : 486
invented-when : 485
has-height : 470
first-product : 443
has-tourist-attraction : 441
has-weight : 427
invented-by : 415
has-highest-mountain : 412
post-code : 278
loc-leader : 251
eats : 105


In [59]:
for _, l in train_set[train_set["label"]=="has-length"].sample(n=5, random_state=6).iterrows():
    print('-'*50)
    print(l["text"])
    e1, rel, e2 = l["entity_1"], l["label"], l["entity_2"]
    print(f"[{e1}] --[{rel}]--> [{e2}]")

--------------------------------------------------
<e1>Gerbils</e1> are typically between <e2>6 and 12 inches</e2> (150 and 300 mm) long, including the tail, which makes up about 1/2 of their total length.
[Gerbils] --[has-length]--> [6 and 12 inches]
--------------------------------------------------
<e1>Irrawaddys</e1> can range from 90 kg (200 lb) to 200 kg (440 lb) and length is <e2>2.3 m</e2> (7.5 ft) at full maturity.
[Irrawaddys] --[has-length]--> [2.3 m]
--------------------------------------------------
<e1>Nautsiyoki River</e1> (Russian: Наутсийоки) is a river in the north of the Kola Peninsula in Murmansk Oblast, Russia. It is <e2>25 kilometers</e2> (16 mi) in length.
[Nautsiyoki River] --[has-length]--> [25 kilometers]
--------------------------------------------------
<e1>Hector's dolphin</e1> (Cephalorhynchus hectori) is the best-known of the four dolphins in the genus Cephalorhynchus and, along with its subspecies Maui's dolphin, is the only cetacean endemic to New Zeala

In [ ]:
MAPPING = {
    "birth-place": [{"subject": "{e1}", "relation": "PerformsAction", "object": "born"}, {"subject" : "born", "relation" : "AtLocation", "object" : "{e2}"}],
    "is-where" : [{"subject": "{e1}", "relation": "AtLocation", "object": "{e2}"}],
    "has-type" : [ {"subject": "{e1}", "relation": "IsA", "object": "{e2}"}],
    "movie-has-director" : [{"subject": "{e1}", "relation": "ReceivesAction", "object": "direct"}, {"subject": "{e2}", "relation": "PerformsAction", "object": "{e2}"}],
    "from-country" : [{"subject": "{e1}", "relation": "IsA", "object": "{e2}"}],
    "has-author" : [{"subject" : "{e1}" , "relation" : "ReceivesAction","object" : "write"}, {"subject" : "{e2}" , "relation" : "PerformsAction","object" : "write"}],
    "has-genre" : [{"subject" : "{e1}" , "relation" : "IsA","object" : "{e2}"}],
    "has-occupation" : [{"subject" : "{e1}" , "relation" : "IsA","object" : "{e2}"}],
    "headquarters" : [{"subject" : "{e1}" , "relation" : "AtLocation","object" : "{e2}"}],
    "org-has-member" : [{"subject" : "{e2}" , "relation" : "PartOf","object" : "{e1}"}],
    "has-population" : [{"subject" : "{e1}" , "relation" : "HasQuantity","object" : "{e2}"}],
    "is-member-of" : [{"subject" : "{e1}" , "relation" : "PartOf","object" : "{e2}"}],
    "org-has-founder" : [{"subject" : "{e2}" , "relation" : "PerformsAction", "object" : "founded"}, {"subject" : "{e1}" , "relation" : "ReceivesAction", "object" : "founded"}],
    "has-parent" : [{"subject" : "{e1}", "relation": "HasA","object" : "parent"}, {"subject" : "{e2}", "relation": "IsA","object" : "parent"}],
    "has-spouse" : [{"subject" : "{e1}", "relation": "HasA","object" : "spouse"}, {"subject" : "{e2}", "relation": "IsA","object" : "spouse"}],
    "won-award" : [{"subject" : "{e1}", "relation": "PerformsAction","object" : "win"}, {"subject" : "{e2}", "relation": "ReceivesAction","object" : "win"}],
    "org-leader" : [{"subject" : "{e2}", "relation": "IsA","object" : "leader"}, {"subject" : "{e2}", "relation": "PartOf","object" : "{e1}"}],
    "has-edu" : [{"subject" : "{e1}", "relation": "PerformsAction","object" : "study"}, {"subject" : "study", "relation": "AtLocation","object" : "{e2}"}],
    "has-nationality" : [{"subject" : "{e1}", "relation": "HasA","object" : "nationality"}, {"subject" : "nationality", "relation": "HasContext","object" : "{e2}"}],
    "starring" : [{"subject" : "{e2}", "relation": "PartOf","object" : "{e1}"}],
    "has-child" : [{"subject" : "{e1}", "relation": "HasA","object" : "child"}, {"subject" : "{e2}", "relation": "IsA","object" : "child"}],
    "event-year" : [],
    "has-sibling" : [],
    "has-length" : [],
    "has-lifespan" : [],
    "invented-when" : [],
    "has-height" : [],
    "first-product" : [],
    "has-tourist-attraction" : [],
    "has-weight" : [],
    "invented-by" : [],
    "has-highest-mountain" : [],
    "post-code" : [],
    "loc-leader" : [],
    "eats" : [{"subject" : "{e1}", "relation": "PerformsAction","object" : "eating"}, {"subject" : "{e2}", "relation": "ReceivesAction","object" : "eating"}]
}
